# EFADT — 01 Data Exploration

Explore the synthetic smart campus dataset.

Generates:
- Occupancy distributions per building
- CO₂ vs occupancy scatter
- Outdoor temperature seasonal profile
- Sensor fault visualization

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Generate a quick 2-day sample for exploration
import pandas as pd
from data.generation.occupancy_model import generate_occupancy_series, generate_co2_from_occupancy
from data.generation.thermal_simulator import simulate_building_thermal_series, generate_hvac_power_series

n = 5760  # 2 days at 30s
ts = pd.date_range('2024-04-15 00:00', periods=n, freq='30s')  # April = exam month
rng = np.random.default_rng(42)
base_rates = np.array([3., 30., 70.])

occ = generate_occupancy_series(n, base_rates, ts, max_occupancy=80, rng=rng)
co2 = generate_co2_from_occupancy(occ, rng=rng)
hvac = generate_hvac_power_series(n, occ, 23.0, 22.0, 25.0, rng=rng)
T_in, T_out = simulate_building_thermal_series(n, occ, hvac, ts, 0.0018, 0.011, 0.009)

df = pd.DataFrame({'timestamp': ts, 'occupancy': occ, 'co2': co2,
                   'T_in': T_in, 'T_out': T_out, 'hvac_kw': hvac})
df.set_index('timestamp', inplace=True)
df.head()

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
df['occupancy'].plot(ax=axes[0], color='steelblue', lw=0.8)
axes[0].set_ylabel('Occupancy (persons)')
axes[0].set_title('Exam Period — 2-Day Campus Sensor Stream (Building B01)')

df['T_in'].plot(ax=axes[1], color='tomato', lw=0.8, label='T_in')
df['T_out'].plot(ax=axes[1], color='orange', lw=0.8, linestyle='--', label='T_out', alpha=0.6)
axes[1].axhline(20, color='gray', lw=0.5, ls=':')
axes[1].axhline(26, color='gray', lw=0.5, ls=':')
axes[1].fill_between(df.index, 20, 26, alpha=0.08, color='green', label='Comfort band')
axes[1].set_ylabel('Temperature (°C)')
axes[1].legend(loc='upper right', fontsize=8)

df['co2'].plot(ax=axes[2], color='purple', lw=0.8)
axes[2].axhline(1000, color='red', lw=0.5, ls='--', label='CO₂ limit')
axes[2].set_ylabel('CO₂ (ppm)')
axes[2].legend(fontsize=8)

df['hvac_kw'].plot(ax=axes[3], color='green', lw=0.8)
axes[3].axhline(0, color='black', lw=0.5)
axes[3].set_ylabel('HVAC Power (kW)')

plt.tight_layout()
plt.savefig('../docs/data_exploration.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Occupancy distribution by hour
df['hour'] = df.index.hour
hourly = df.groupby('hour')['occupancy'].agg(['mean', 'std'])

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(hourly.index, hourly['mean'], yerr=hourly['std'],
       color='steelblue', alpha=0.8, capsize=3)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Mean Occupancy (persons)')
ax.set_title('Hourly Occupancy Profile — Exam Period')
ax.set_xticks(range(0, 24, 2))
plt.tight_layout()
plt.show()
print(f'Peak occupancy at hour {hourly["mean"].idxmax()}:00 — {hourly["mean"].max():.1f} persons')